In [1]:
import os
import sys
import re
import glob
import math
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

random.seed(0)
np.random.seed(0)

try:
    import torch  # noqa: F401

    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    device = "cpu"

device



'cuda'

In [2]:
BASE_DIR = "/kaggle/data/rsna-2022-cervical-spine-fracture-detection"

TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
TEST_CSV = os.path.join(BASE_DIR, "test.csv")
SAMPLE_SUB = os.path.join(BASE_DIR, "sample_submission.csv")

TRAIN_IMAGES_PATH = os.path.join(BASE_DIR, "train_images")
TEST_IMAGES_PATH = os.path.join(BASE_DIR, "test_images")
IMAGES_DIR = TEST_IMAGES_PATH

assert os.path.exists(TRAIN_CSV), f"Missing {TRAIN_CSV}"
assert os.path.exists(TEST_CSV), f"Missing {TEST_CSV}"
assert os.path.exists(SAMPLE_SUB), f"Missing {SAMPLE_SUB}"



In [3]:
segmentation_checkpoint = (
    "../input/effdet-models/axial_segmentation_effseg_132508-epoch-100.pth"
)
axial_det_checkpoint1 = (
    "../input/effdet-models/axial_detection_effdet_134352-epoch-52.pth"
)
axial_det_checkpoint2 = (
    "../input/effdet-models/axial_detection_effdet_001015-epoch-150.pth"
)
axial_yolo_checkpoint1 = "../input/effdet-models/yolo_custom4_epoch_099.pt"



In [4]:
yolo_model = None
try:
    yolo_path = "../input/effdet-models/yolov7"
    if os.path.isdir(yolo_path):
        sys.path.append(yolo_path)
        from models.experimental import attempt_load  # type: ignore

        import torch

        yolo_model = attempt_load(axial_yolo_checkpoint1, map_location=device).eval()
except Exception as e:
    yolo_model = None
    yolo_import_error = str(e)

yolo_model is not None



False

In [5]:
df_test = pd.read_csv(TEST_CSV)

df_test_slices = (
    df_test[["StudyInstanceUID"]]
    .drop_duplicates()
    .assign(Slice=0, Start=0)
    .reset_index(drop=True)
)

df_test_slices.head()



,StudyInstanceUID,Slice,Start
0,1.2.826.0.1.3680043.6200,0,0
1,1.2.826.0.1.3680043.27262,0,0
2,1.2.826.0.1.3680043.12351,0,0
3,1.2.826.0.1.3680043.1363,0,0
4,1.2.826.0.1.3680043.4859,0,0


In [6]:
df_test_slices = df_test_slices.copy()
df_test_slices.head()




,StudyInstanceUID,Slice,Start
0,1.2.826.0.1.3680043.6200,0,0
1,1.2.826.0.1.3680043.27262,0,0
2,1.2.826.0.1.3680043.12351,0,0
3,1.2.826.0.1.3680043.1363,0,0
4,1.2.826.0.1.3680043.4859,0,0


In [7]:
def rescale_img_to_hu(dcm_ds):
    return dcm_ds.pixel_array * dcm_ds.RescaleSlope + dcm_ds.RescaleIntercept


def normalize_hu(data):
    data = np.clip(data, a_min=-2242, a_max=2242) / 4484 + 0.5
    return data


def normalize_hu_t(data):
    return np.clip(data, a_min=-2242.0, a_max=2242.0) / 2242.0


def normalize_hu_tensor(data):
    import torch

    return torch.clip(data, min=-2242.0, max=2242.0) / 2242.0


def load_dicom(path):
    import pydicom as dicom

    ds = dicom.dcmread(path)
    img = rescale_img_to_hu(ds)
    return img, float(ds.PixelSpacing[0])




In [8]:
try:
    import torch
    import torchvision.transforms as T
    import torchvision.transforms.functional as TF
    from torch.utils.data import DataLoader, Dataset
except Exception:
    torch = None
    T = None
    TF = None
    DataLoader = None
    Dataset = object


class DcmDataSet(Dataset):
    def __init__(self, df, path, image_size=512):
        self.df = df
        self.path = path
        self.len = len(self.df)
        self.image_size = image_size
        self.transform = T.Resize((image_size, image_size)) if T is not None else None

    def __getitem__(self, i):
        if torch is None:
            raise RuntimeError(
                "Torch/torchvision not available for DICOM dataset reading."
            )
        s = self.df.iloc[i]
        gpath = os.path.join(self.path, s.StudyInstanceUID, f"{int(s.Slice)}.dcm")
        g, pixel_spacing = load_dicom(gpath)
        g = normalize_hu_t(g)
        x = torch.as_tensor(g, dtype=torch.float32).unsqueeze(0)
        if self.transform is not None:
            x = self.transform(x)
        return x, pixel_spacing, bool(s.Slice == s.Start)

    def __len__(self):
        return self.len


ds = None  # not used in fallback



In [9]:
batch_size = 16
dl = None



In [10]:
seg_model = None
try:
    effunet_path = "../input/effdet-models/efficientunet-pytorch-0.0.6"
    if os.path.isdir(effunet_path):
        sys.path.append(effunet_path)
        from efficientunet import get_efficientunet_b5  # type: ignore

        def get_axial_segmentation_model(checkpoint):
            import torch

            model = get_efficientunet_b5(
                out_channels=2, concat_input=True, pretrained=False
            )
            state = torch.load(checkpoint, map_location=torch.device(device))
            model.load_state_dict(state["model"])
            model.eval()
            return model.to(device)

        if os.path.exists(segmentation_checkpoint):
            seg_model = get_axial_segmentation_model(segmentation_checkpoint)
except Exception as e:
    seg_model = None
    seg_import_error = str(e)

seg_model is not None



False

In [11]:
effdet_models = []
try:
    effdet_path = "../input/effdet-models/effdet"
    if os.path.isdir(effdet_path):
        sys.path.append(effdet_path)
        from effdet import create_model  # type: ignore

        import torch

        def get_axial_detection_model(checkpoint, image_size=512):
            model = create_model(
                "efficientdetv2_ds",
                bench_task="predict",
                num_classes=1,
                image_size=(image_size, image_size),
                pretrained=False,
                max_det_per_image=1,
            )
            state = torch.load(checkpoint, map_location=torch.device(device))
            model.load_state_dict(state["model"])
            return model.eval().to(device)

        if os.path.exists(axial_det_checkpoint1):
            effdet_models = [get_axial_detection_model(axial_det_checkpoint1)]
except Exception as e:
    effdet_models = []
    effdet_import_error = str(e)

effdet_models



[]

In [12]:
IMAGE_SIZES = [640, 512]
det_model_names = ["yolo", "effdet"]
det_models = ([yolo_model] if yolo_model is not None else []) + effdet_models




In [13]:
def get_axial_boundary_from_segmentation(
    seg, pixel_spacing, throw=100, tol=0.2, max_mm=100
):
    import torch

    image_size = seg.shape[0]
    min_size = min(image_size, max_mm / pixel_spacing)

    rows, columns = seg.nonzero(as_tuple=True)
    rows.sort()
    columns.sort()

    throw = min(len(rows) // 2, throw)

    if (len(rows)) == 0:
        return torch.tensor([0, 0, image_size, image_size]).to(seg.device)

    xmin, xmax = columns[throw], columns[-throw]
    ymin, ymax = rows[throw], rows[-throw]

    w = (xmax - xmin) * (1 + tol)
    h = (ymax - ymax) * (1 + tol)
    new_size = max(w, h, min_size)
    new_size = min(image_size, new_size)

    xcenter, ycenter = (xmax + xmin) / 2, (ymax + ymin) / 2

    xmin = torch.min(
        torch.tensor(image_size - new_size, device=seg.device), xcenter - new_size / 2
    )
    xmin = xmin.clip(min=0)

    ymin = torch.min(
        torch.tensor(image_size - new_size, device=seg.device), ycenter - new_size / 2
    )
    ymin = ymin.clip(min=0)

    return torch.stack([xmin, ymin, xmin + new_size, ymin + new_size])




In [14]:
def predict_seg(x, model, seg_img_size=256):
    import torchvision.transforms.functional as TF

    x = TF.resize(x, (seg_img_size, seg_img_size))
    logits = model(x)
    classification_score, mse_score = logits.sigmoid().chunk(2, dim=1)
    classification_pred = classification_score.gt(0.5).float()
    pred = classification_pred * mse_score
    return pred




In [15]:
def get_axial_boundary_from_seg(segs, pixel_spacings, seg_img_size=256):
    import torch

    boundary_list = []
    for i in range(segs.shape[0]):
        seg = segs[i, 0, :, :]
        boundary = get_axial_boundary_from_segmentation(
            seg,
            float(pixel_spacings[i]),
            throw=int(100.0 / 512.0 * seg_img_size),
            tol=0.2,
            max_mm=100.0 / 512.0 * seg_img_size,
        )
        boundary_list.append(boundary)
    boundary_list = torch.stack(boundary_list, axis=0) * (512.0 / seg_img_size)
    return boundary_list




In [16]:
def convert_yolo_result(pred):
    import torch

    max_indices = torch.argmax(pred[:, :, 4], dim=1)
    max_values = pred[torch.arange(pred.shape[0]), max_indices, :]
    bboxes, scores = max_values[:, :4], max_values[:, 4]
    bboxes[:, 2] += bboxes[:, 0]
    bboxes[:, 3] += bboxes[:, 1]
    return bboxes, scores


def predict_det(x, model):
    pred_result = model(x)
    if isinstance(pred_result, tuple) is True:
        return convert_yolo_result(pred_result[0])
    else:
        return pred_result[:, 0, :4], pred_result[:, 0, 4]


def crop_resize_images(imgs_tensor, boundary_list, img_size=512):
    import torchvision.transforms.functional as TF
    import torch

    croped_list = []
    for i in range(imgs_tensor.shape[0]):
        xmin, ymin, xmax, ymax = boundary_list[i, :]
        xmin, ymin, xmax, ymax = int(xmin), int(ymin), int(xmax), int(ymax)
        croped = TF.crop(
            imgs_tensor[i, :, :, :],
            top=ymin,
            left=xmin,
            height=ymax - ymin,
            width=xmax - xmin,
        )
        croped = TF.resize(croped, (img_size, img_size))
        croped_list.append(croped)
    return torch.stack(croped_list, 0)


def get_original_bbox(bbox, boundary, image_size=512.0):
    scale = image_size / (boundary[:, [2]] - boundary[:, [0]])
    org_bbox = bbox / scale
    org_bbox[:, 0] += boundary[:, 0]
    org_bbox[:, 1] += boundary[:, 1]
    org_bbox[:, 2] += boundary[:, 0]
    org_bbox[:, 3] += boundary[:, 1]
    return org_bbox




In [17]:
def get_bbox_class(seg, bbox):
    import torch

    xmin, ymin, xmax, ymax = bbox.int()
    area = seg[ymin:ymax, xmin:xmax]
    result = torch.mean(area[area > 0])
    result = torch.round(result / 0.125)
    return result




In [18]:
def get_bbox_class_list(seg_list, seg_bboxes):
    import torch

    class_list = []
    for i in range(seg_list.shape[0]):
        class_index = get_bbox_class(seg_list[i, :, :], seg_bboxes[i, :])
        class_list.append(class_index)
    return torch.stack(class_list)




In [19]:
def get_class_score(scores, class_list, eps=1e-2):
    import torch

    result = scores.new_zeros((scores.shape[0], 8)) + eps
    class_list = torch.nan_to_num(class_list).long()
    result[torch.arange(scores.shape[0]), class_list] = scores
    return result


def check_detection_result(det_result, img_size=512.0, threshold=0.2):
    import torch

    areas = (
        (det_result[:, 2] - det_result[:, 0])
        * (det_result[:, 3] - det_result[:, 1])
        / (img_size * img_size)
    )
    big_indices = torch.argwhere(areas > threshold)
    det_result[big_indices, 4] = 0.0
    return det_result




In [20]:
def cal_loss(prob, label):
    pos_weight = np.array([14, 2, 2, 2, 2, 2, 2, 2])
    neg_weight = np.array([7, 1, 1, 1, 1, 1, 1, 1])
    score = pos_weight * label * np.log(prob) + neg_weight * (1 - label) * np.log(
        1 - prob
    )
    weight_total = pos_weight * label + neg_weight * (1 - label)
    return -score.sum(axis=1) / weight_total.sum(axis=1)




In [21]:
def compute_leave_one_out_prevalences(
    train_csv: str, clip_value: float = 5e-4, alpha: float = 6.0, beta: float = 24.0
) -> pd.DataFrame:
    df_train = pd.read_csv(train_csv)
    target_cols = ["patient_overall"] + [f"C{i}" for i in range(1, 8)]

    n = len(df_train)
    sums = df_train[target_cols].sum(axis=0).astype(float)

    loo = pd.DataFrame(
        index=df_train["StudyInstanceUID"], columns=target_cols, dtype=float
    )
    denom = (n - 1) + alpha + beta
    denom = max(float(denom), 1.0)

    for c in target_cols:
        loo[c] = (sums[c] - df_train[c].astype(float).values + alpha) / denom

    loo = loo.clip(clip_value, 1 - clip_value)
    loo.index.name = "StudyInstanceUID"
    return loo


def predict_patient_level_from_prevalence(
    train_csv: str, clip_value: float = 5e-4, alpha: float = 6.0, beta: float = 24.0
) -> pd.DataFrame:
    df_train = pd.read_csv(train_csv)
    target_cols = ["patient_overall"] + [f"C{i}" for i in range(1, 8)]

    n = len(df_train)
    sums = df_train[target_cols].sum(axis=0).astype(float)

    prevalences = (sums + alpha) / (float(n) + alpha + beta)
    prevalences = prevalences.clip(clip_value, 1 - clip_value)
    return pd.DataFrame([prevalences.values], columns=target_cols)


CLIP_VALUE = 5e-4
ALPHA = 6.0
BETA = 24.0

df_prev = predict_patient_level_from_prevalence(
    TRAIN_CSV, clip_value=CLIP_VALUE, alpha=ALPHA, beta=BETA
)
df_loo = compute_leave_one_out_prevalences(
    TRAIN_CSV, clip_value=CLIP_VALUE, alpha=ALPHA, beta=BETA
)

df_prev, df_loo.head()




(   patient_overall        C1        C2        C3        C4       C5        C6  \
 0         0.431034  0.112069  0.150862  0.064655  0.073276  0.12069  0.172414   
 
          C7  
 0  0.168103  ,
                            patient_overall        C1        C2        C3  \
 StudyInstanceUID                                                           
 1.2.826.0.1.3680043.21561         0.428571  0.112554  0.147186  0.064935   
 1.2.826.0.1.3680043.9447          0.428571  0.112554  0.147186  0.064935   
 1.2.826.0.1.3680043.29194         0.432900  0.112554  0.151515  0.064935   
 1.2.826.0.1.3680043.21586         0.428571  0.108225  0.147186  0.064935   
 1.2.826.0.1.3680043.19266         0.432900  0.112554  0.151515  0.064935   
 
                                  C4        C5       C6        C7  
 StudyInstanceUID                                                  
 1.2.826.0.1.3680043.21561  0.073593  0.121212  0.17316  0.168831  
 1.2.826.0.1.3680043.9447   0.073593  0.121212  0.17316  0

In [22]:
def set_patient_overall_from_any(
    df_probs: pd.DataFrame, clip_value: float = 5e-4
) -> pd.DataFrame:
    df_probs = df_probs.copy()
    c_cols = [f"C{i}" for i in range(1, 8)]
    p_any = 1.0 - np.prod(1.0 - df_probs[c_cols].values, axis=1)
    p_any = np.clip(p_any, clip_value, 1 - clip_value)
    df_probs["patient_overall"] = p_any.astype(float)
    return df_probs


def shrink_probs_toward_half(p: np.ndarray, temperature: float = 1.35) -> np.ndarray:
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, CLIP_VALUE, 1.0 - CLIP_VALUE)
    logit = np.log(p / (1.0 - p))
    logit = logit / float(temperature)
    out = 1.0 / (1.0 + np.exp(-logit))
    return np.clip(out, CLIP_VALUE, 1.0 - CLIP_VALUE)


def blend_patient_prior_and_any(
    p_prior: np.ndarray, p_any: np.ndarray, lam: float, clip_value: float
) -> np.ndarray:
    p_prior = np.clip(
        np.asarray(p_prior, dtype=np.float64), clip_value, 1.0 - clip_value
    )
    p_any = np.clip(np.asarray(p_any, dtype=np.float64), clip_value, 1.0 - clip_value)
    lam = float(lam)
    out = np.exp((1.0 - lam) * np.log(p_prior) + lam * np.log(p_any))
    return np.clip(out, clip_value, 1.0 - clip_value)


TEMP = 1.35  # keep existing vertebra calibration setting

# CHANGE (score direction: lower-is-better, current worse than target):
# Reduce influence of p_any (derived from shrunk vertebra probs) on the heavily-weighted patient_overall,
# keeping patient_overall closer to the smoothed/LOO patient prior to reduce weighted logloss.
PATIENT_BLEND_LAMBDA = 0.01



In [23]:
df_test_studies = df_test[["StudyInstanceUID"]].drop_duplicates().reset_index(drop=True)

target_cols = ["patient_overall"] + [f"C{i}" for i in range(1, 8)]
c_cols = [f"C{i}" for i in range(1, 8)]

df_patient_pred = df_test_studies.copy().set_index("StudyInstanceUID")
for c in target_cols:
    df_patient_pred[c] = float(df_prev.iloc[0][c])

overlap_uids = df_patient_pred.index.intersection(df_loo.index)
if len(overlap_uids) > 0:
    df_patient_pred.loc[overlap_uids, target_cols] = df_loo.loc[
        overlap_uids, target_cols
    ]

df_patient_pred[c_cols] = shrink_probs_toward_half(
    df_patient_pred[c_cols].values, temperature=TEMP
)

p_any = 1.0 - np.prod(1.0 - df_patient_pred[c_cols].values, axis=1)
p_any = np.clip(p_any, CLIP_VALUE, 1.0 - CLIP_VALUE)
p_prior = np.clip(
    df_patient_pred["patient_overall"].values.astype(np.float64),
    CLIP_VALUE,
    1.0 - CLIP_VALUE,
)

df_patient_pred["patient_overall"] = blend_patient_prior_and_any(
    p_prior, p_any, lam=PATIENT_BLEND_LAMBDA, clip_value=CLIP_VALUE
).astype(float)

df_patient_pred.head()



,patient_overall,C1,C2,C3,C4,C5,C6,C7
StudyInstanceUID,,,,,,,,
1.2.826.0.1.3680043.6200,0.433528,0.17753,0.217568,0.12141,0.132438,0.186783,0.238315,0.234237
1.2.826.0.1.3680043.27262,0.433528,0.17753,0.217568,0.12141,0.132438,0.186783,0.238315,0.234237
1.2.826.0.1.3680043.12351,0.433528,0.17753,0.217568,0.12141,0.132438,0.186783,0.238315,0.234237
1.2.826.0.1.3680043.1363,0.433528,0.17753,0.217568,0.12141,0.132438,0.186783,0.238315,0.234237
1.2.826.0.1.3680043.4859,0.433528,0.17753,0.217568,0.12141,0.132438,0.186783,0.238315,0.234237


In [24]:
pass



In [25]:
df_test = pd.read_csv(TEST_CSV)
df_test.head()



,StudyInstanceUID,prediction_type,row_id
0,1.2.826.0.1.3680043.6200,patient_overall,1.2.826.0.1.3680043.6200_patient_overall
1,1.2.826.0.1.3680043.27262,patient_overall,1.2.826.0.1.3680043.27262_patient_overall
2,1.2.826.0.1.3680043.12351,patient_overall,1.2.826.0.1.3680043.12351_patient_overall
3,1.2.826.0.1.3680043.1363,patient_overall,1.2.826.0.1.3680043.1363_patient_overall
4,1.2.826.0.1.3680043.4859,patient_overall,1.2.826.0.1.3680043.4859_patient_overall


In [26]:
df_sub = df_test.set_index("StudyInstanceUID").join(df_patient_pred, how="left")

fill_map = df_prev.iloc[0].to_dict()
for c in target_cols:
    df_sub[c] = df_sub[c].fillna(fill_map[c]).astype(float)

p_any_row = 1.0 - np.prod(1.0 - df_sub[c_cols].to_numpy(dtype=np.float64), axis=1)
p_any_row = np.clip(p_any_row, CLIP_VALUE, 1.0 - CLIP_VALUE)
p_prior_row = np.clip(
    df_sub["patient_overall"].to_numpy(dtype=np.float64), CLIP_VALUE, 1.0 - CLIP_VALUE
)

df_sub["patient_overall"] = blend_patient_prior_and_any(
    p_prior_row, p_any_row, lam=PATIENT_BLEND_LAMBDA, clip_value=CLIP_VALUE
)

pred_types = df_sub["prediction_type"].astype(str).to_numpy()
col_idx = pd.Index(target_cols).get_indexer(pred_types)
if (col_idx < 0).any():
    bad = np.unique(pred_types[col_idx < 0]).tolist()
    raise ValueError(f"Unexpected prediction_type values not in target_cols: {bad}")

prob_matrix = df_sub[target_cols].to_numpy(dtype=np.float64)
df_sub["fractured"] = prob_matrix[np.arange(len(df_sub)), col_idx]
df_sub["fractured"] = np.clip(
    df_sub["fractured"].to_numpy(dtype=np.float64), CLIP_VALUE, 1.0 - CLIP_VALUE
)

df_sub = df_sub.reset_index(drop=False)
df_sub[["row_id", "fractured"]].head()



,row_id,fractured
0,1.2.826.0.1.3680043.6200_patient_overall,0.43601
1,1.2.826.0.1.3680043.27262_patient_overall,0.43601
2,1.2.826.0.1.3680043.12351_patient_overall,0.43601
3,1.2.826.0.1.3680043.1363_patient_overall,0.43601
4,1.2.826.0.1.3680043.4859_patient_overall,0.43601


In [27]:
sub_path = "submission.csv"
df_sub[["row_id", "fractured"]].to_csv(sub_path, index=False)

df_sample = pd.read_csv(SAMPLE_SUB)
assert list(df_sample.columns) == ["row_id", "fractured"]
assert df_sub.shape[0] == df_sample.shape[0], "Row count mismatch vs sample submission"
assert set(df_sub["row_id"]) == set(
    df_sample["row_id"]
), "row_id set mismatch vs sample submission"
print(f"Wrote {sub_path} with shape {df_sub[['row_id','fractured']].shape}")

Wrote submission.csv with shape (14536, 2)
